In [ ]:
!pip install datasets transformers evaluate sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [1]:
import pandas as pd
import re

In [3]:
df = pd.read_csv('/content/drive/MyDrive/psychometrics-bot/mbti.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8675 entries, 0 to 8674
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   type    8675 non-null   object
 1   posts   8675 non-null   object
dtypes: object(2)
memory usage: 135.7+ KB


In [8]:
df['type'].unique()

array(['INFJ', 'ENTP', 'INTP', 'INTJ', 'ENTJ', 'ENFJ', 'INFP', 'ENFP',
       'ISFP', 'ISTP', 'ISFJ', 'ISTJ', 'ESTP', 'ESFP', 'ESTJ', 'ESFJ'],
      dtype=object)

In [9]:
df['type'].value_counts()

,count
type,
INFP,1832
INFJ,1470
INTP,1304
INTJ,1091
ENTP,685
ENFP,675
ISTP,337
ISFP,271
ENTJ,231


In [76]:
logging.basicConfig(level=logging.INFO, force=True)

In [81]:
"""
MBTI personality type classifier.
Trains on English labeled data (type + posts), predicts 4 dichotomies (I/E, N/S, T/F, P/J).

Usage:
  - Train: see train_mbti.py or clf.train(csv_path, output_dir)
  - Predict (English): clf.load(model_path); mbti, conf = clf.predict(text)
  - Predict (Russian users): use translator.prepare_text_for_mbti(text) then clf.predict(...)
  - Confidence dict: per-dichotomy probability (e.g. conf["N/S"] ≈ 0.5 → suggest questions for N/S)
"""

import logging
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, force=True)

# MBTI: 4 dichotomies, multi-label (one binary per dimension)
MBTI_TRAITS = ["I/E", "N/S", "T/F", "P/J"]
TRAIT_LETTERS = [("I", "E"), ("N", "S"), ("T", "F"), ("P", "J")]


def type_to_labels(mbti_type: str) -> list[float]:
    """Convert 4-letter MBTI type to 4 binary labels (1 = first letter of pair)."""
    mbti_type = (mbti_type or "").strip().upper()
    if len(mbti_type) != 4:
        raise ValueError(f"Invalid MBTI type: {mbti_type!r}")
    return [
        1.0 if mbti_type[i] == TRAIT_LETTERS[i][0] else 0.0
        for i in range(4)
    ]


def labels_to_type(labels: list[float]) -> str:
    """Convert 4 binary labels or probabilities to 4-letter MBTI type."""
    return "".join(
        TRAIT_LETTERS[i][0] if (labels[i] >= 0.5) else TRAIT_LETTERS[i][1]
        for i in range(4)
    )


def _preprocess_text(text: str) -> str:
        if not isinstance(text, str) or len(text) == 0:
            return ""

        html_pattern = re.compile(r"<[^>]+>")
        url_pattern = re.compile(r"https?://[^\s]+")
        mail_pattern = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")
        phone_pattern = re.compile(r"[\+]?[0-9\s\-\(\)]{10,}")
        multiple_spaces = re.compile(r"\s+")
        dates_pattern = re.compile(
            r"\d{1,2}\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4}"
        )

        # убираем html теги
        text = html_pattern.sub("", text)

        # заменяем ссылки
        text = url_pattern.sub("", text)
        # заменяем почту
        text = mail_pattern.sub("", text)

        # заменяем телефоны
        text = phone_pattern.sub("", text)

        # заменяем даты
        text = dates_pattern.sub("", text)

        # приводим к lowercase
        text = text.lower()

        # заменяем множественные пробелы на один пробел
        text = multiple_spaces.sub(" ", text)

        # удаляем лишние знаки (но оставляем точки, запятые)
        text = re.sub(r"[^\w\s\.,!?]", " ", text)

        text = text.strip()
        return text


class MBTIClassifier:
    """
    Multi-label classifier for MBTI (4 dichotomies).
    Expects English text; for Russian, translate first (e.g. translator.translate_ru_to_en).
    """

    def __init__(
        self,
        model_name: str = "distilbert-base-uncased",
        max_length: int = 512,
        device: Optional[torch.device] = None,
    ):
        self.model_name = model_name
        self.max_length = max_length
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=4,
            problem_type="multi_label_classification",
        )
        self.model = self.model.to(self.device)
        logger.info(f"MBTI classifier loaded on {self.device}")

    def load_data(
        self,
        csv_path: str,
        text_column: str = "posts",
        type_column: str = "type",
    ) -> tuple[list[str], list[list[float]]]:
        """Load MBTI CSV with columns type_column and text_column. Returns (texts, list of 4 labels)."""
        path = Path(csv_path)
        if not path.exists():
            raise FileNotFoundError(f"Dataset not found: {csv_path}")

        df = pd.read_csv(path)
        if type_column not in df.columns or text_column not in df.columns:
            raise ValueError(
                f"CSV must have columns {type_column!r} and {text_column!r}; got {list(df.columns)}"
            )

        texts = []
        labels = []
        for _, row in df.iterrows():
            t = (row[text_column] or "").strip()
            if not t or len(t) < 20:
                continue
            try:
                lab = type_to_labels(str(row[type_column]).strip())
            except (ValueError, TypeError):
                continue
            texts.append(_preprocess_text(t))
            labels.append(lab)

        logger.info(f"Loaded {len(texts)} samples from {csv_path}")
        return texts, labels

    def train(
        self,
        csv_path: str,
        output_dir: str,
        num_epochs: int = 5,
        batch_size: int = 8,
        val_size: float = 0.2,
        lr: float = 2e-5,  # Lowered slightly for full-model fine-tuning
        patience: int = 3,
        text_column: str = "posts",
        type_column: str = "type",
        freeze_encoder_epochs: int = 0, # Unfreeze from the start
    ) -> dict:
        """
        Train on CSV and save best model to output_dir.
        Returns dict with eval metrics (e.g. macro_f1, exact_match_accuracy).

        Why metrics can stay dead (all 0s, F1=0):
        1) Gradient sanitization: when we zero out NaN/Inf in grads and still step, the
           remaining gradient is often biased toward "predict 0" (the batches that didn't
           explode were often the ones with more 0s). So the model never learns to predict 1s.
        2) LayerNorm explosion: some batches cause huge grads → we skip or zero them → we
           lose the learning signal for those batches (often the diverse ones that need 1s).
        3) Data: "posts" in MBTI datasets are usually long concatenations (many Reddit posts).
           We truncate to max_length tokens (default 512), so we only see the start of each user's text.
           If the MBTI signal is in the full pattern, not the first 256 tokens, the model
           has nothing to learn from.

        What we do: skip the step only when a *trainable* parameter has non-finite gradient
        (frozen encoder can have NaN grads from LayerNorm; we ignore those). Optionally
        freeze the encoder for the first N epochs so only the head is trained (no backbone
        update → no LayerNorm corruption). Try freeze_encoder_epochs=1 and/or longer
        max_length if you have memory.
        """
        texts, labels = self.load_data(csv_path, text_column=text_column, type_column=type_column)
        if not texts:
            return {"error": "no valid samples"}

        labels_arr = np.array(labels)
        n_samples = len(labels_arr)
        # Per-trait label distribution (for logging)
        trait_ratios = [labels_arr[:, j].mean() for j in range(4)]
        logger.info(f"Dataset size: {n_samples}")
        logger.info(f"Label balance (fraction of 1s per trait I/E, N/S, T/F, P/J): {[f'{r:.3f}' for r in trait_ratios]}")
        logger.info(f"Count of 1s per trait: {[int(labels_arr[:, j].sum()) for j in range(4)]}")

        # Stratify by full type so val set has similar distribution
        types = [labels_to_type(list(l)) for l in labels]
        try:
            train_idx, val_idx = train_test_split(
                range(len(texts)),
                test_size=val_size,
                random_state=42,
                stratify=types,
            )
        except ValueError:
            train_idx, val_idx = train_test_split(
                range(len(texts)),
                test_size=val_size,
                random_state=42,
            )
        train_idx, val_idx = np.array(train_idx), np.array(val_idx)

        # Balanced sample weights per dimension (like sklearn class_weight="balanced"):
        # Weight_ij = n/(2*n_class) so minority class matters equally AND loss scale stays ~0.5 (no NaN/tiny grads).
        n_train = len(train_idx)

        train_labels_np = labels_arr[train_idx]

        # Calculate pos_weight for BCEWithLogitsLoss: (Number of Negatives) / (Number of Positives)
        pos_weights = []
        for j in range(4):
            n_pos = max(1, int(train_labels_np[:, j].sum()))
            n_neg = max(1, len(train_labels_np) - n_pos)
            pos_weights.append(n_neg / n_pos)

        pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(self.device)
        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
        logger.info(f"Using pos_weights for BCE (I/E, N/S, T/F, P/J): {[round(w, 4) for w in pos_weights]}")

        encodings = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors="pt",
        )
        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]
        labels_t = torch.tensor(labels, dtype=torch.float32)

        train_ids = input_ids[train_idx]
        val_ids = input_ids[val_idx]
        train_mask = attention_mask[train_idx]
        val_mask = attention_mask[val_idx]
        train_labels = labels_t[train_idx]
        val_labels = labels_t[val_idx]

        logger.info(f"Train size: {len(train_ids)}, Val size: {len(val_ids)}")
        logger.info(f"Val label distribution (count of 1s per trait): {[int(val_labels[:, j].sum()) for j in range(4)]}")

        # Removed train_weights_t from the dataset
        train_ds = TensorDataset(
            train_ids.to(self.device),
            train_mask.to(self.device),
            train_labels.to(self.device),
        )
        val_ds = TensorDataset(
            val_ids.to(self.device),
            val_mask.to(self.device),
            val_labels.to(self.device),
        )
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size)

        def _set_encoder_frozen(frozen: bool) -> None:
            # When frozen=True: only classifier (and pooler) trainable; encoder frozen.
            for name, p in self.model.named_parameters():
                p.requires_grad = (not frozen) or ("classifier" in name or "pooler" in name)

        if freeze_encoder_epochs > 0:
            _set_encoder_frozen(True)
            trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
            logger.info(f"Encoder frozen for first {freeze_encoder_epochs} epoch(s). Trainable params: {trainable}")

        optimizer = torch.optim.AdamW(
            [p for p in self.model.parameters() if p.requires_grad],
            lr=lr,
            weight_decay=0.01,
        )
        total_steps = len(train_loader) * num_epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps,
        )

        best_f1 = 0.0
        best_exact = 0.0
        best_state = None
        no_improve = 0

        from sklearn.metrics import f1_score

        for epoch in range(num_epochs):
            # Unfreeze encoder after freeze_encoder_epochs; recreate optimizer with all params.
            if freeze_encoder_epochs > 0 and epoch == freeze_encoder_epochs:
                _set_encoder_frozen(False)
                unfreeze_lr = lr * 0.5  # often safer to use smaller lr when unfreezing
                optimizer = torch.optim.AdamW(self.model.parameters(), lr=unfreeze_lr, weight_decay=0.01)
                remaining_steps = (num_epochs - freeze_encoder_epochs) * len(train_loader)
                scheduler = get_linear_schedule_with_warmup(
                    optimizer,
                    num_warmup_steps=int(0.1 * remaining_steps),
                    num_training_steps=remaining_steps,
                )
                logger.info(f"Encoder unfrozen from epoch {epoch+1}; lr={unfreeze_lr}")

            self.model.train()
            train_loss_sum = 0.0
            num_batches = 0
            num_skipped = 0
            LOGITS_CLAMP = 20.0

            # ---> THIS IS THE ONLY BATCH LOOP YOU SHOULD HAVE <---
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
                iids, mask, lab = batch
                optimizer.zero_grad()

                out = self.model(input_ids=iids, attention_mask=mask)
                logits = out.logits

                logits = torch.where(torch.isfinite(logits), logits, torch.zeros_like(logits, device=logits.device))
                logits = logits.clamp(-LOGITS_CLAMP, LOGITS_CLAMP)

                loss = criterion(logits, lab)

                # Check loss validity BEFORE backward pass
                if not torch.isfinite(loss):
                    num_skipped += 1
                    continue

                loss.backward()

                # 1. Gather only trainable parameters
                trainable_params = [p for p in self.model.parameters() if p.requires_grad]

                # 2. Check for ANY non-finite gradients
                has_non_finite_grad = False
                for p in trainable_params:
                    if p.grad is not None and not torch.isfinite(p.grad).all():
                        has_non_finite_grad = True
                        break

                # 3. Skip the step if we found NaNs/Infs
                if has_non_finite_grad:
                    num_skipped += 1
                    continue

                # 4. If gradients are clean, proceed with clipping and stepping
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)

                optimizer.step()
                scheduler.step()

                train_loss_sum += loss.item()
                num_batches += 1

                if epoch == 0 and num_batches == 1:
                    with torch.no_grad():
                        logits_mean = out.logits.float().mean().item()
                        logits_std = out.logits.float().std().item()
                        probs_mean = torch.sigmoid(out.logits).mean().item()
                    logger.info(f"First valid batch: loss={loss.item():.4f}, logits mean={logits_mean:.4f} std={logits_std:.4f}, probs mean={probs_mean:.4f}")

            # ---> VALIDATION PHASE STARTS IMMEDIATELY AFTER THE BATCH LOOP FINISHES <---
            avg_train_loss = train_loss_sum / num_batches if num_batches else float("nan")
            if num_skipped > 0:
                logger.info(f"Epoch {epoch+1} — Skipped {num_skipped} batches (non-finite loss or gradients).")

            # Validation
            self.model.eval()
            all_preds = []
            all_labels = []
            with torch.no_grad():
                for iids, mask, lab in val_loader:
                    out = self.model(input_ids=iids, attention_mask=mask)
                    probs = torch.sigmoid(out.logits)
                    all_preds.append((probs > 0.5).long().cpu())
                    all_labels.append(lab.cpu())
            preds = torch.cat(all_preds, dim=0).numpy()
            labels_np = torch.cat(all_labels, dim=0).numpy()

            pred_counts = [int(preds[:, j].sum()) for j in range(4)]
            label_counts = [int(labels_np[:, j].sum()) for j in range(4)]
            logger.info(f"Epoch {epoch+1} — Train loss: {avg_train_loss:.4f}")
            logger.info(f"Epoch {epoch+1} — Val: pred 1s per trait (I/E, N/S, T/F, P/J): {pred_counts} (total val samples: {len(preds)})")
            logger.info(f"Epoch {epoch+1} — Val: true 1s per trait: {label_counts}")

            per_label_f1 = []
            for j in range(4):
                f1_j = f1_score(labels_np[:, j], preds[:, j], average="binary", zero_division=0)
                per_label_f1.append(f1_j)
            macro_f1 = float(np.mean(per_label_f1))
            exact = np.all(preds == labels_np, axis=1).mean()

            logger.info(f"Epoch {epoch+1} — Per-trait F1 (I/E, N/S, T/F, P/J): {[f'{x:.4f}' for x in per_label_f1]}")
            logger.info(f"Epoch {epoch+1} — macro_f1: {macro_f1:.4f}, exact_match: {exact:.4f}")
            if macro_f1 > best_f1:
                best_f1 = macro_f1
                best_exact = exact
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                no_improve = 0
                logger.info(f"New best model (macro_f1={macro_f1:.4f}).")
            else:
                no_improve += 1
                logger.info(f"No improvement ({no_improve}/{patience}).")
            if no_improve >= patience:
                logger.info(f"Early stopping at epoch {epoch + 1}")
                break

        if best_state:
            self.model.load_state_dict(best_state)
            logger.info("Loaded best model state from validation.")

        out_path = Path(output_dir)
        out_path.mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(out_path / "model")
        self.tokenizer.save_pretrained(out_path / "tokenizer")
        logger.info(f"Model saved to {output_dir}")

        return {"macro_f1": float(best_f1), "exact_match_accuracy": float(best_exact)}

    def predict(
        self,
        text: str,
        return_confidence: bool = True,
    ) -> tuple[str, Optional[dict]]:
        """
        Predict MBTI type from English text.
        Returns (mbti_type, confidence_dict or None).
        confidence_dict: per dichotomy probabilities, e.g. {"I/E": 0.8, "N/S": 0.3, ...}.
        """
        text = _preprocess_text(text or "")
        if not text or len(text) < 10:
            return "INFP", None  # fallback; you may prefer to raise or return None

        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
            padding=True,
        ).to(self.device)

        with torch.no_grad():
            out = self.model(**inputs)

        logits = out.logits[0].cpu()
        probs = torch.sigmoid(logits).numpy()
        # 1 = first letter (I, N, T, P), 0 = second (E, S, F, J)
        pred_labels = [1 if p >= 0.5 else 0 for p in probs]
        mbti_type = "".join(
            TRAIT_LETTERS[i][pred_labels[i]] for i in range(4)
        )

        if return_confidence:
            conf = {}
            for i, name in enumerate(MBTI_TRAITS):
                conf[name] = float(probs[i])  # prob of first letter
            return mbti_type, conf
        return mbti_type, None

    def load(self, path: str) -> None:
        """Load model and tokenizer from directory (must contain model/ and tokenizer/)."""
        path = Path(path)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            path / "model",
            num_labels=4,
            problem_type="multi_label_classification",
        )
        self.tokenizer = AutoTokenizer.from_pretrained(path / "tokenizer")
        self.model = self.model.to(self.device)
        logger.info(f"MBTI model loaded from {path}")

    def save(self, path: str) -> None:
        """Save model and tokenizer to directory."""
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(path / "model")
        self.tokenizer.save_pretrained(path / "tokenizer")
        logger.info(f"MBTI model saved to {path}")



In [ ]:
def predict_mbti(
    text: str,
    model_path: str,
    translate_russian: bool = True,
) -> tuple[str, Optional[dict]]:
    """
    One-shot prediction: load model, optionally translate Russian → English, predict.
    Returns (mbti_type, confidence_dict).
    """
    if translate_russian:
        try:
            from translator import prepare_text_for_mbti
            text = prepare_text_for_mbti(text)
        except Exception as e:
            logger.warning("Translation skipped: %s", e)
    clf = MBTIClassifier()
    clf.load(model_path)
    return clf.predict(text, return_confidence=True)

In [82]:
classifier = MBTIClassifier()

INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/vocab.txt "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/vocab.txt "HTTP/1.1 200 OK"


...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/special_tokens_map.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
INFO:__main__:MBTI classifier loaded on cuda


In [1]:
classifier.train(csv_path='/content/drive/MyDrive/psychometrics-bot/mbti.csv', output_dir='/content/mbti_classificator')

NameError: name 'classifier' is not defined

In [18]:
!pwd

/content


In [51]:
texts, labels = classifier.load_data(csv_path='/content/drive/MyDrive/psychometrics-bot/mbti.csv')

INFO:__main__:Loaded 8675 samples from /content/drive/MyDrive/psychometrics-bot/mbti.csv


In [63]:
labels[1]

[0.0, 1.0, 1.0, 1.0]

In [22]:
texts[0]

"' url and intj moments url sportscenter not top ten plays url pranks|||what has been the most life-changing experience in your life?||| url url on repeat for most of today.|||may the perc experience immerse you.|||the last thing my infj friend posted on his facebook before committing suicide the next day. rest in peace~ url enfj7. sorry to hear of your distress. it's only natural for a relationship to not be perfection all the time in every moment of existence. try to figure the hard times as times of growth, as...|||84389 84390 url url ...|||welcome and stuff.||| url game. set. match.|||prozac, wellbrutin, at least thirty minutes of moving your legs (and i don't mean moving them while sitting in your same desk chair), weed in moderation (maybe try edibles as a healthier alternative...|||basically come up with three items you've determined that each type (or whichever types you want to do) would more than likely use, given each types' cognitive functions and whatnot, when left by...||

In [52]:
texts[0]

'and intj moments sportscenter not top ten plays pranks   what has been the most life changing experience in your life?    on repeat for most of today.   may the perc experience immerse you.   the last thing my infj friend posted on his facebook before committing suicide the next day. rest in peace  enfj7. sorry to hear of your distress. it s only natural for a relationship to not be perfection all the time in every moment of existence. try to figure the hard times as times of growth, as...   ...   welcome and stuff.    game. set. match.   prozac, wellbrutin, at least thirty minutes of moving your legs  and i don t mean moving them while sitting in your same desk chair , weed in moderation  maybe try edibles as a healthier alternative...   basically come up with three items you ve determined that each type  or whichever types you want to do  would more than likely use, given each types  cognitive functions and whatnot, when left by...   all things in moderation. sims is indeed a video 

In [57]:
lengths = [len(text.split()) for text in texts]
mean_length = sum(lengths)/len(lengths)

In [58]:
mean_length

1304.5854755043229

In [68]:
"""
Russian → English translation for MBTI inference.
Dataset is in English; when the user writes in Russian, we translate to English
before calling the MBTI classifier.
"""

import logging
from typing import Optional

logger = logging.getLogger(__name__)

# Lazy-loaded pipeline to avoid loading heavy model at import time
_translation_pipeline = None


def _get_pipeline():
    global _translation_pipeline
    if _translation_pipeline is None:
        try:
            from transformers import pipeline
            _translation_pipeline = pipeline(
                "translation",
                model="Helsinki-NLP/opus-mt-ru-en",
                device=-1,  # CPU; use 0 for GPU if available
            )
            logger.info("Loaded Helsinki-NLP/opus-mt-ru-en translation model")
        except Exception as e:
            logger.warning("Translation model not available: %s", e)
            raise
    return _translation_pipeline


def translate_ru_to_en(text: str) -> str:
    """
    Translate Russian text to English using Helsinki-NLP/opus-mt-ru-en.
    If text is empty or translation fails, returns original text.
    """
    if not text or not str(text).strip():
        return ""

    text = str(text).strip()
    try:
        pipe = _get_pipeline()
        out = pipe(text, max_length=512, truncation=True)
        if out and isinstance(out, list) and len(out) > 0:
            return (out[0].get("translation_text") or text).strip()
        return text
    except Exception as e:
        logger.warning("Translation failed: %s", e)
        return text


def is_likely_russian(text: str) -> bool:
    """Heuristic: True if text contains Cyrillic characters."""
    if not text:
        return False
    return bool(__import__("re").search(r"[\u0400-\u04FF]", str(text)))


def prepare_text_for_mbti(text: str) -> str:
    """
    If text looks Russian, translate to English; otherwise return as-is.
    Use this before calling MBTIClassifier.predict() for Russian users.
    """
    if not text or not str(text).strip():
        return ""
    return translate_ru_to_en(text) if is_likely_russian(text) else str(text).strip()


In [70]:
"""
Simple MBTI classifier: TF-IDF + Logistic Regression (one binary model per trait).
No GPU, no gradients, no NaN. Same predict(type, confidence) interface as the neural classifier.
"""

import logging
import re
from pathlib import Path
from typing import Optional

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# from translator import prepare_text_for_mbti

logger = logging.getLogger(__name__)

MBTI_TRAITS = ["I/E", "N/S", "T/F", "P/J"]
TRAIT_LETTERS = [("I", "E"), ("N", "S"), ("T", "F"), ("P", "J")]


def type_to_labels(mbti_type: str) -> list[float]:
    mbti_type = (mbti_type or "").strip().upper()
    if len(mbti_type) != 4:
        raise ValueError(f"Invalid MBTI type: {mbti_type!r}")
    return [
        1.0 if mbti_type[i] == TRAIT_LETTERS[i][0] else 0.0
        for i in range(4)
    ]


def labels_to_type(labels: list[float]) -> str:
    return "".join(
        TRAIT_LETTERS[i][0] if (labels[i] >= 0.5) else TRAIT_LETTERS[i][1]
        for i in range(4)
    )


def _preprocess_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+", " URL ", text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


class MBTIClassifierSimple:
    """
    Four binary Logistic Regression classifiers on TF-IDF features.
    Same interface as MBTIClassifier: load_data, train, predict, save, load.
    """

    def __init__(
        self,
        max_features: int = 20_000,
        ngram_range: tuple[int, int] = (1, 2),
        C: float = 1.0,
    ):
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.C = C
        self.vectorizer: Optional[TfidfVectorizer] = None
        self.models: list[LogisticRegression] = []

    def load_data(
        self,
        csv_path: str,
        text_column: str = "posts",
        type_column: str = "type",
    ) -> tuple[list[str], list[list[float]]]:
        path = Path(csv_path)
        if not path.exists():
            raise FileNotFoundError(f"Dataset not found: {csv_path}")
        df = pd.read_csv(path)
        if type_column not in df.columns or text_column not in df.columns:
            raise ValueError(f"CSV must have {type_column!r} and {text_column!r}")
        texts = []
        labels = []
        for _, row in df.iterrows():
            t = (row[text_column] or "").strip()
            if not t or len(t.split()) < 20:
                continue
            try:
                lab = type_to_labels(str(row[type_column]).strip())
            except (ValueError, TypeError):
                continue
            texts.append(_preprocess_text(t))
            labels.append(lab)
        logger.info(f"Loaded {len(texts)} samples from {csv_path}")
        return texts, labels

    def train(
        self,
        csv_path: str,
        output_dir: str,
        val_size: float = 0.2,
        text_column: str = "posts",
        type_column: str = "type",
    ) -> dict:
        texts, labels = self.load_data(csv_path, text_column=text_column, type_column=type_column)
        if not texts:
            return {"error": "no valid samples"}

        labels_arr = np.array(labels)
        for j in range(4):
            r = labels_arr[:, j].mean()
            logger.info(f"Trait {MBTI_TRAITS[j]} fraction of 1s: {r:.3f}")

        # Stratify by full type
        types = [labels_to_type(list(l)) for l in labels]
        try:
            train_idx, val_idx = train_test_split(
                range(len(texts)),
                test_size=val_size,
                random_state=42,
                stratify=types,
            )
        except ValueError:
            train_idx, val_idx = train_test_split(
                range(len(texts)), test_size=val_size, random_state=42
            )

        X_train = [texts[i] for i in train_idx]
        X_val = [texts[i] for i in val_idx]
        y_train = labels_arr[train_idx]
        y_val = labels_arr[val_idx]

        self.vectorizer = TfidfVectorizer(
            max_features=self.max_features,
            ngram_range=self.ngram_range,
            min_df=2,
            max_df=0.95,
            strip_accents="unicode",
        )
        X_train_tf = self.vectorizer.fit_transform(X_train)
        X_val_tf = self.vectorizer.transform(X_val)

        self.models = []
        per_f1 = []
        for j in range(4):
            clf = LogisticRegression(
                class_weight="balanced",
                C=self.C,
                max_iter=500,
                random_state=42,
            )
            clf.fit(X_train_tf, y_train[:, j])
            self.models.append(clf)
            pred_j = clf.predict(X_val_tf)
            f1_j = f1_score(y_val[:, j], pred_j, average="binary", zero_division=0)
            per_f1.append(f1_j)
        macro_f1 = float(np.mean(per_f1))
        preds = np.column_stack([self.models[j].predict(X_val_tf) for j in range(4)])
        exact = np.all(preds == y_val, axis=1).mean()

        logger.info(f"Per-trait F1: {[f'{x:.4f}' for x in per_f1]}")
        logger.info(f"macro_f1: {macro_f1:.4f}, exact_match: {exact:.4f}")

        out_path = Path(output_dir)
        out_path.mkdir(parents=True, exist_ok=True)
        self.save(str(out_path))
        logger.info(f"Model saved to {output_dir}")

        return {"macro_f1": macro_f1, "exact_match_accuracy": float(exact)}

    def predict(
        self,
        text: str,
        return_confidence: bool = True,
    ) -> tuple[str, Optional[dict]]:
        text = _preprocess_text(text or "")
        if not text or len(text.split()) < 5:
            return "INFP", None

        if self.vectorizer is None or len(self.models) != 4:
            raise RuntimeError("Model not loaded or not trained")

        text = prepare_text_for_mbti(str(text).strip())

        X = self.vectorizer.transform([text])
        probs = np.array([self.models[j].predict_proba(X)[0, 1] for j in range(4)])
        pred_labels = [1 if p >= 0.5 else 0 for p in probs]
        mbti_type = "".join(TRAIT_LETTERS[i][pred_labels[i]] for i in range(4))

        if return_confidence:
            conf = {MBTI_TRAITS[i]: float(probs[i]) for i in range(4)}
            return mbti_type, conf
        return mbti_type, None

    def save(self, path: str) -> None:
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        joblib.dump(
            {"vectorizer": self.vectorizer, "models": self.models},
            path / "mbti_simple.joblib",
        )
        logger.info(f"Simple MBTI model saved to {path}")

    def load(self, path: str) -> None:
        path = Path(path)
        data = joblib.load(path / "mbti_simple.joblib")
        self.vectorizer = data["vectorizer"]
        self.models = data["models"]
        logger.info(f"Simple MBTI model loaded from {path}")


In [66]:
classifier = MBTIClassifierSimple()
classifier.train(csv_path='/content/drive/MyDrive/psychometrics-bot/mbti.csv', output_dir='/content/mbti_classificator')

INFO:__main__:Loaded 8674 samples from /content/drive/MyDrive/psychometrics-bot/mbti.csv
INFO:__main__:Trait I/E fraction of 1s: 0.770
INFO:__main__:Trait N/S fraction of 1s: 0.862
INFO:__main__:Trait T/F fraction of 1s: 0.459
INFO:__main__:Trait P/J fraction of 1s: 0.604
INFO:__main__:Per-trait F1: ['0.9018', '0.9347', '0.8463', '0.8332']
INFO:__main__:macro_f1: 0.8790, exact_match: 0.5689
INFO:__main__:Simple MBTI model saved to /content/mbti_classificator
INFO:__main__:Model saved to /content/mbti_classificator


{'macro_f1': 0.8790164908478478, 'exact_match_accuracy': 0.5688760806916426}

In [73]:
classifier.predict('I really like to hang out with friends, we go with them to walks every day actually')

('INFJ',
 {'I/E': 0.47204446075769246,
  'N/S': 0.3944501752737424,
  'T/F': 0.6282940085039881,
  'P/J': 0.5650427340664971})

In [74]:
"""
Test examples of user self-descriptions for each MBTI type.
Use these to check that the model returns plausible predictions.

Run:
  python test_mbti_examples.py
  # or in Python:
  from test_mbti_examples import EXAMPLES, run_examples
  run_examples()  # needs a loaded classifier
"""

# Short self-descriptions (20+ words) that stereotypically fit each type.
# Format: (text, expected_type). The model may not match exactly; these are for sanity checks.
EXAMPLES = [
    # I/E
    (
        "I love spending weekends alone with a book or a project. Too much socializing drains me. "
        "I think before I speak and prefer deep conversations with one or two people over small talk at parties. "
        "I need quiet time to recharge.",
        "INFP",
    ),
    (
        "I get energy from being around people. I love meeting new folks and I think out loud. "
        "I'm always up for a party or a spontaneous hangout. I get bored and restless when I'm alone too long. "
        "I'm the one who plans group trips and keeps the chat going.",
        "ENFP",
    ),
    # N/S
    (
        "I'm always thinking about possibilities and what could be. I love abstract ideas and big-picture thinking. "
        "I get bored with too many routine details. I trust my intuition about people and situations. "
        "I'd rather discuss theories than stick to step-by-step plans.",
        "INTP",
    ),
    (
        "I'm very practical. I focus on what's real and what works. I notice details and remember facts. "
        "I prefer clear instructions and concrete examples over abstract theories. I like hands-on work "
        "and getting things done in the here and now.",
        "ISTJ",
    ),
    # T/F
    (
        "I make decisions with my head. I value logic and fairness over harmony. I can seem cold but I just "
        "prefer to analyze things objectively. I'd rather be right than liked. I get frustrated when people "
        "make choices based only on feelings.",
        "INTJ",
    ),
    (
        "I care a lot about how people feel. I'm tuned into others' emotions and I want everyone to get along. "
        "I make decisions based on my values and how it affects people. I sometimes put others' needs before "
        "my own. I hate conflict and try to find win-win solutions.",
        "INFJ",
    ),
    # P/J
    (
        "I like to keep my options open. I'm flexible and go with the flow. I often do things at the last minute "
        "and I'm okay with that. I prefer exploring over sticking to a strict plan. I get bored with too much "
        "structure and routine.",
        "ENFP",
    ),
    (
        "I like to plan ahead and have a clear schedule. I make lists and get things done early. I feel stressed "
        "when things are disorganized or when people are late. I prefer closure and decided outcomes. "
        "I'm reliable and people know they can count on me to follow through.",
        "ESTJ",
    ),
    # Mix of types – full 16
    (
        "I'm quiet, logical, and independent. I have strong opinions and I don't care much about social norms. "
        "I like to understand how things work and I can be very focused on my interests. I prefer working alone "
        "and I'm not very emotional in my decisions.",
        "INTP",
    ),
    (
        "I'm outgoing and love helping people. I'm warm, organized, and I notice what others need. "
        "I like to create harmony and I take my commitments seriously. I'm the person who remembers birthdays "
        "and makes sure everyone feels included.",
        "ESFJ",
    ),
    (
        "I'm creative and idealistic. I march to my own drum and I care deeply about my values. "
        "I see possibilities everywhere and I want to understand people. I can be sensitive and I need meaning "
        "in what I do. I don't like conflict or strict rules.",
        "INFP",
    ),
    (
        "I'm direct and decisive. I like to lead and get results. I'm strategic and I don't sugarcoat things. "
        "I have high standards for myself and others. I get impatient with inefficiency. "
        "I'd rather take charge than wait for someone else to act.",
        "ENTJ",
    ),
]



Usage: python test_mbti_examples.py [model_path]
Model path not found: -f


In [75]:
for ex in EXAMPLES:
   print(f'ACTUAL TYPE: {ex[1]}, PREDICTED_TYPE: {classifier.predict(ex[0])}')
   print('-'*100)

ACTUAL TYPE: INFP, PREDICTED_TYPE: ('ESFJ', {'I/E': 0.6103477700124855, 'N/S': 0.5189835082445761, 'T/F': 0.5104983999289259, 'P/J': 0.5189750355700457})
----------------------------------------------------------------------------------------------------
ACTUAL TYPE: ENFP, PREDICTED_TYPE: ('INFJ', {'I/E': 0.3201140948709782, 'N/S': 0.3951107521800337, 'T/F': 0.551978879815213, 'P/J': 0.6511824911485502})
----------------------------------------------------------------------------------------------------
ACTUAL TYPE: INTP, PREDICTED_TYPE: ('ESFJ', {'I/E': 0.5180502708428889, 'N/S': 0.6544135164034653, 'T/F': 0.6370429250544263, 'P/J': 0.6274133473820511})
----------------------------------------------------------------------------------------------------
ACTUAL TYPE: ISTJ, PREDICTED_TYPE: ('ENFJ', {'I/E': 0.6523866385572574, 'N/S': 0.2566001285079218, 'T/F': 0.6119539664134888, 'P/J': 0.5579102224451573})
----------------------------------------------------------------------------------